# Lab 2: Convolutional Neural Network (CNN)




In this laboratory we continue to work with Keras. We will focus on Convolutional Neural Network
we are going to work with cifar10, a  dataset consisting of 60000 32x32 colour images in 10 classes, with 6000 images per class. There are 50000 training images and 10000 test images.
Therefore the main goal of this laboratory is to solve a multiclass classification problem with 10 different classes


### Loading the dataset 

In [1]:
import numpy as np
import tensorflow as tf 
import os
new_im_size = 32
channels = 3

(train_X,train_Y),(test_X,test_Y) = tf.keras.datasets.cifar10.load_data()


c:\Users\Mario\Documents\UNIGE\Unige_code\Master\First\MSC_py_vem\Lib\site-packages\keras\src\datasets\cifar.py:18: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  d = cPickle.load(f, encoding="bytes")


# 2.1 Dataset pre-processing
The first thing that we need to do when we are dealing with a new dataset is to operate some pre-processing operations. Data preprocessing usually refers to the steps applied to make data more suitable for learning. 
In this section we are going to deal with:

* 2.1.1 Normalization
* 2.1.2 Standardization
* 2.1.3 Splitting and label preprocessing


## 2.1.1 Normalization
One common practice in training a Neural Network is to normalize the images by dividing each pixel value by the maximum value that we can have, i.e. 255.<br>
The purpose of this is to obtain a mean close to 0.<br>
Normalizing the data generally speeds up learning and leads to faster convergence

In [2]:
# Normalizing the data
print("Normalizing training set..")
train_X = np.asarray(train_X, dtype=np.float32) / 255 # Normalizing training set
print("Normalizing test set..")
test_X = np.asarray(test_X, dtype=np.float32) / 255 # Normalizing test set

Normalizing training set..
Normalizing test set..


## 2.1.2 Standardization
Another common practice in data pre-processing is standardization.<br>
The idea about standardization is to compute your dataset mean and standard deviation in order to subtract from every data point $x$ the dataset mean $\mu$ and then divide by the standard deviation $\sigma$.<br>
The outcome of this operation is to obtain a distribution with mean equal to 0 and a standard deviation equal to 1.<br>
By applying normalization to our data we are making the features more similar to each other and this usually makes the learning process easier.<br>


In [3]:
# Standardizing the data
def standardize_dataset(X):
    image_means = []
    image_stds = []

    for image in X:
        image_means.append(np.mean(image)) # Computing the image mean
        image_stds.append(np.std(image)) # Computing the image standard deviation

    dataset_mean = np.mean(image_means) # Computing the dataset mean
    dataset_std = np.mean(image_stds) # Computing the dataset standard deviation
    return [dataset_mean, dataset_std] # For every image we subtract to it the dataset mean and we divide by the dataset standard deviation


In [4]:
dataset_mean, dataset_std = standardize_dataset(train_X)
print("Standardizing training set..")
train_X = (train_X-dataset_mean)/dataset_std # Standardizing the training set
print("Standardizing test set..")
test_X = (test_X-dataset_mean)/dataset_std # Standardizing the test set

Standardizing training set..
Standardizing test set..


## 2.1.3 Splitting and label preprocessing
Now we just need to split our training set in orer to get the validation set and convert our labels to one-hot representation

In [5]:
# Creating the validation set
from sklearn.model_selection import train_test_split
print("Splitting training set to create validation set..")
train_X, valid_X, train_Y, valid_Y = train_test_split(train_X, train_Y, test_size=0.2, random_state=13)

# Converting labels to one-hot representation
from tensorflow.keras.utils import to_categorical
train_Y_one_hot = to_categorical(train_Y) # Converting training labels to one-hot representation
valid_Y_one_hot = to_categorical(valid_Y) # Converting validation labels to one-hot representation
test_Y_one_hot = to_categorical(test_Y) # Converting test labels to one-hot representation

print("Size of training")
print(len(train_X))
print(len(train_Y_one_hot))

print("Size of validation")
print(len(valid_X))
print(len(valid_Y_one_hot))

Splitting training set to create validation set..
Size of training
40000
40000
Size of validation
10000
10000


# 2.2 Training a model from scratch
Now that we have properly pre-processed our data, we are going to create a convolutional model in Keras. 
Usually a convolutional model is made by two subsequent part:
* A convolutional part
* A fully connected

Usually the convolutional part is made by some layers composed by
* convolutional layer: performs a spatial convolution over images
* pooling layer: used to reduce the output spatial dimension from $n$ to 1 by averaging the $n$ different value or considering the maximum between them 
* dropout layer: applied to a layer, consists of randomly "dropping out" (i.e. set to zero) a number of output features of the layer during training.

The convolutional part produces its output and the fully connected part ties together the received information in order to solve the classification problem.
Let us start with a shallow architecture with only 2 conv

In [6]:
# Creating the model from scratch
import tensorflow.keras
from tensorflow.keras import Sequential,Input,Model
from tensorflow.keras.layers import Dense, Dropout, Flatten
from tensorflow.keras.layers import Conv2D, MaxPooling2D
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.layers import LeakyReLU
from sklearn.metrics import accuracy_score

# Network parameters
batch_size = 32 # Setting the batch size
epochs = 10 # Setting the number of epochs
num_classes = 10 # Getting the amount of classes
print(num_classes)
scratch_model = Sequential()	

# Build here your keras model.
# Try to use one convolutional layer, joint with pooling layer and dropout layer

# Creating conv 1: conv with 32 kernels of size 3x3, padding='same', input_shape=(new_im_size, new_im_size, channels) 
# + LeakyReLU(alpha=0.1) + maxpooling with region size 2x2 and padding=''
scratch_model.add(Conv2D(32, kernel_size=(3, 3), padding='same', input_shape=(new_im_size, new_im_size, channels)))
scratch_model.add(LeakyReLU(alpha=0.1))
scratch_model.add(MaxPooling2D(pool_size=(2, 2), padding='valid'))

# Adding the dense final part: Flatten + Dense with 64 neurons and relu + Dropout 25% + Dense with 10 neurons and softmax
scratch_model.add(Flatten())
scratch_model.add(Dense(64, activation='relu'))
scratch_model.add(Dropout(0.25))
scratch_model.add(Dense(10, activation='softmax'))

# Compile the model with the Adam optimizer
scratch_model.compile(loss=tensorflow.keras.losses.categorical_crossentropy, optimizer=tensorflow.keras.optimizers.Adam(),metrics=['accuracy'])

# Visualize the model through the summary function
scratch_model.summary()

10


c:\Users\Mario\Documents\UNIGE\Unige_code\Master\First\MSC_py_vem\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
c:\Users\Mario\Documents\UNIGE\Unige_code\Master\First\MSC_py_vem\Lib\site-packages\keras\src\layers\activations\leaky_relu.py:41: UserWarning: Argument `alpha` is deprecated. Use `negative_slope` instead.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 32, 32, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu (LeakyReLU)         │ (None, 32, 32, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 16, 16, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 8192)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │       524,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 525,898 (2.01 MB)

 Trainable params: 525,898 (2.01 MB)

 Non-trainable params: 0 (0.00 B)

In [7]:
# Let's train the model!
scratch_model_history = scratch_model.fit(train_X, train_Y_one_hot, batch_size=batch_size, shuffle=True, epochs=epochs, validation_data=(valid_X, valid_Y_one_hot))

Epoch 1/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.3885 - loss: 1.6578 - val_accuracy: 0.5309 - val_loss: 1.3271
Epoch 2/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.4967 - loss: 1.3740 - val_accuracy: 0.5760 - val_loss: 1.1972
Epoch 3/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.5414 - loss: 1.2531 - val_accuracy: 0.6048 - val_loss: 1.1321
Epoch 4/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.5702 - loss: 1.1781 - val_accuracy: 0.6176 - val_loss: 1.0959
Epoch 5/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.5941 - loss: 1.1188 - val_accuracy: 0.6324 - val_loss: 1.0560
Epoch 6/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 6s 5ms/step - accuracy: 0.6149 - loss: 1.0642 - val_accuracy: 0.6230 - val_loss: 1.0660
Epoch 7/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.6307 - loss: 1.0179 - val_accuracy: 0.6308 - val_loss: 1.0547
Epoch 8/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6491 - loss: 0.9632 - 

In [8]:
# Getting the results
scratch_model_train_acc = scratch_model_history.history['accuracy']
scratch_model_valid_acc = scratch_model_history.history['val_accuracy']
scratch_model_train_loss = scratch_model_history.history['loss']
scratch_model_valid_loss = scratch_model_history.history['val_loss']

print("Test accuracy: ", accuracy_score(np.argmax(scratch_model.predict(test_X), axis=-1), test_Y))

313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
Test accuracy:  0.6429


**Is the obtained value coherent with what you expected?**<br>


### Try to make the network deeper, adding more Conv and Pooling layers. Do the performances improve? 

In [9]:
# Network parameters
batch_size = 32 # Setting the batch size
epochs = 10 # Setting the number of epochs
num_classes = 10# Getting the amount of classes
deeper_model = Sequential()	

# Build here your keras model.
# Try to use one or more convolutional layer, joint with pooling layer and dropout layer

# Creating conv 1 with 32 kernels of size 3x3, LeakyReLU with alpha=0.1, max pooling and 25% dropout 
deeper_model.add(Conv2D(32, kernel_size=(3, 3), padding='same', input_shape=(new_im_size, new_im_size, channels)))
deeper_model.add(LeakyReLU(alpha=0.1))
deeper_model.add(MaxPooling2D(pool_size=(2, 2)))
deeper_model.add(Dropout(0.25))

# Creating conv 2 with 64 kernels of size 3x3, LeakyReLU with alpha=0.1, max pooling and 25% dropout 
deeper_model.add(Conv2D(64, kernel_size=(3, 3), padding='same'))
deeper_model.add(LeakyReLU(alpha=0.1))
deeper_model.add(MaxPooling2D(pool_size=(2, 2)))
deeper_model.add(Dropout(0.25))

# Adding the dense final part: 
# - flatten
# a dense layer with 64 neurons, LeakyReLU with alpha=0.1, 25% dropout
# a dense layer with <num_classes> neurons and softmax activation
deeper_model.add(Flatten())
deeper_model.add(Dense(64))
deeper_model.add(LeakyReLU(alpha=0.1))
deeper_model.add(Dropout(0.25))
deeper_model.add(Dense(num_classes, activation='softmax'))

# Compile the model with the Adam optimizer
deeper_model.compile(loss=tensorflow.keras.losses.categorical_crossentropy, optimizer=tensorflow.keras.optimizers.Adam(),metrics=['accuracy'])

# Visualize the model through the summary function
deeper_model.summary()

c:\Users\Mario\Documents\UNIGE\Unige_code\Master\First\MSC_py_vem\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
c:\Users\Mario\Documents\UNIGE\Unige_code\Master\First\MSC_py_vem\Lib\site-packages\keras\src\layers\activations\leaky_relu.py:41: UserWarning: Argument `alpha` is deprecated. Use `negative_slope` instead.
  warnings.warn(


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_1 (Conv2D)               │ (None, 32, 32, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_1 (LeakyReLU)       │ (None, 32, 32, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 16, 16, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 16, 16, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 16, 16, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_2 (LeakyReLU)       │ (None, 16, 16, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 8, 8, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 8, 8, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 4096)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │       262,208 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_3 (LeakyReLU)       │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 282,250 (1.08 MB)

 Trainable params: 282,250 (1.08 MB)

 Non-trainable params: 0 (0.00 B)

In [10]:
deeper_model_history = deeper_model.fit(train_X, train_Y_one_hot, batch_size=batch_size, shuffle=True, epochs=epochs, validation_data=(valid_X, valid_Y_one_hot))

Epoch 1/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 9s 7ms/step - accuracy: 0.4895 - loss: 1.4366 - val_accuracy: 0.6209 - val_loss: 1.0781
Epoch 2/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 8s 6ms/step - accuracy: 0.6167 - loss: 1.0879 - val_accuracy: 0.6555 - val_loss: 0.9858
Epoch 3/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 8s 7ms/step - accuracy: 0.6558 - loss: 0.9766 - val_accuracy: 0.6869 - val_loss: 0.8838
Epoch 4/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 8s 7ms/step - accuracy: 0.6817 - loss: 0.8994 - val_accuracy: 0.7001 - val_loss: 0.8804
Epoch 5/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 8s 7ms/step - accuracy: 0.7007 - loss: 0.8494 - val_accuracy: 0.7141 - val_loss: 0.8281
Epoch 6/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 8s 7ms/step - accuracy: 0.7152 - loss: 0.8094 - val_accuracy: 0.7032 - val_loss: 0.8622
Epoch 7/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 8s 7ms/step - accuracy: 0.7217 - loss: 0.7847 - val_accuracy: 0.7128 - val_loss: 0.8391
Epoch 8/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 8s 7ms/step - accuracy: 0.7326 - loss: 0.7505 - 

In [11]:
# Getting the results
deeper_model_train_acc = deeper_model_history.history['accuracy']
deeper_model_valid_acc = deeper_model_history.history['val_accuracy']
deeper_model_train_loss = deeper_model_history.history['loss']
deeper_model_valid_loss = deeper_model_history.history['val_loss']

print("Test accuracy: ", accuracy_score(np.argmax(deeper_model.predict(test_X), axis=-1), test_Y))

313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
Test accuracy:  0.7254


# 2.3 Using a pre-trained model
A common alternative to train a model from scratch consists in using a pre-trained model.<br>
The idea is to replace the convolutional part with a highly optimized convolutional part engineered and trained previously by someone else.<br>
Usually the models that we can use through keras.applications have been trained over the image net dataset. <br>
Today we are going to use the Xception Net model. The structure of the architecture can be found here: https://drive.google.com/file/d/1I7d0jc4TiLffDCrcwi_UZ9gwozae5GMe/view?usp=sharing 
<br>
After the convolutional part replacement we still need to set up a fully connected part.<br>
**Why in this lab we cannot use the fully connected part of Xception Net?<br>
What should we do to use it?<br>



In [12]:
# Creating the model based over the pretrained Xception network
from tensorflow.keras import applications
import tensorflow
from tensorflow.keras import models
from tensorflow.keras import layers
from tensorflow.keras import optimizers
model = models.Sequential()

model.add(tensorflow.keras.layers.UpSampling2D(size=(7,7),input_shape=(32,32,3)))

Xception_model = applications.Xception(weights = "imagenet", include_top=False, input_shape = (224, 224, channels))

for layer in Xception_model.layers:
    layer.trainable = False
    
Inputs = layers.Input(shape=(32,32,3))
x = model(Inputs)
x = Xception_model(x)
x = layers.Flatten()(x)
# let's add a fully-connected layer with 128 neurons and ReLU
x = layers.Dense(128, activation='relu')(x)
# and an output layer for 10 classes and softmax activation
predictions = layers.Dense(10, activation='softmax')(x)

# this is the model we will train
pre_trained_model = tensorflow.keras.Model(Inputs, outputs=predictions)
pre_trained_model.compile(loss=tensorflow.keras.losses.categorical_crossentropy, optimizer=tensorflow.keras.optimizers.Adam(),metrics=['accuracy'])

c:\Users\Mario\Documents\UNIGE\Unige_code\Master\First\MSC_py_vem\Lib\site-packages\keras\src\layers\reshaping\up_sampling2d.py:72: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [13]:
# Visualize the model through the summary function
pre_trained_model.summary()

Model: "functional_21"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_4 (InputLayer)      │ (None, 32, 32, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential_2 (Sequential)       │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ xception (Functional)           │ (None, 7, 7, 2048)     │    20,861,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 100352)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │    12,845,184 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 33,707,954 (128.59 MB)

 Trainable params: 12,846,474 (49.01 MB)

 Non-trainable params: 20,861,480 (79.58 MB)

In [14]:
# Let's train the model!
pretrained_model_history = pre_trained_model.fit(train_X, train_Y_one_hot, epochs=epochs, batch_size=batch_size, validation_data=(valid_X, valid_Y_one_hot))

Epoch 1/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 1462s 1s/step - accuracy: 0.7154 - loss: 0.9699 - val_accuracy: 0.7607 - val_loss: 0.7012
Epoch 2/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 1453s 1s/step - accuracy: 0.8086 - loss: 0.5630 - val_accuracy: 0.7715 - val_loss: 0.6829
Epoch 3/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 1456s 1s/step - accuracy: 0.8472 - loss: 0.4410 - val_accuracy: 0.7923 - val_loss: 0.6529
Epoch 4/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 1454s 1s/step - accuracy: 0.8795 - loss: 0.3496 - val_accuracy: 0.7777 - val_loss: 0.7447
Epoch 5/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 1456s 1s/step - accuracy: 0.8998 - loss: 0.2858 - val_accuracy: 0.7933 - val_loss: 0.7555
Epoch 6/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 1549s 1s/step - accuracy: 0.9161 - loss: 0.2371 - val_accuracy: 0.7668 - val_loss: 0.9396
Epoch 7/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 1902s 2s/step - accuracy: 0.9321 - loss: 0.1945 - val_accuracy: 0.7876 - val_loss: 0.8837
Epoch 8/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 1739s 1s/step - accuracy: 0.9416 -

In [16]:
# Getting the results
pretrained_model_train_acc = pretrained_model_history.history['accuracy']
pretrained_model_valid_acc = pretrained_model_history.history['val_accuracy']
pretrained_model_train_loss = pretrained_model_history.history['loss']
pretrained_model_valid_loss = pretrained_model_history.history['val_loss']

test_X_feature = pre_trained_model.predict(test_X)# Producing the test feature
print("Test accuracy: ", accuracy_score(pre_trained_model.predict_classes(test_X_feature), test_Y)) # Testing the model

313/313 ━━━━━━━━━━━━━━━━━━━━ 435s 1s/step


AttributeError: 'Functional' object has no attribute 'predict_classes'

# 2.4 Comparing the models

Now that we trained both the "from scratch" and the "pre-trained" models, we are going to compare the obtained results obtained during the training. We are going to consider accuracy and loss.<br>
**What can you expect from these plots?**

In [ ]:
# Create here the plots to compare the "from scratch" model and the "pretrained" model
# Try to produce a comparison plot about the accuracies (train and validation) and another plot for the losses
# Creating the plots to compare the "from scratch" model and the "pretrained" model
# Producing accuracy over epochs plot

import matplotlib.pyplot as plt

# Assuming the models have been trained and history objects are available
# history_scratch, history_deeper, history_pretrained

# Create a figure with two subplots side by side
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Plot 1: Accuracy comparison
ax1.plot(history_scratch.history['accuracy'], label='Scratch - Train')
ax1.plot(history_scratch.history['val_accuracy'], label='Scratch - Val')
ax1.plot(history_deeper.history['accuracy'], label='Deeper - Train')
ax1.plot(history_deeper.history['val_accuracy'], label='Deeper - Val')
ax1.plot(history_pretrained.history['accuracy'], label='Pretrained - Train')
ax1.plot(history_pretrained.history['val_accuracy'], label='Pretrained - Val')
ax1.set_title('Model Accuracy Comparison')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(True)

# Plot 2: Loss comparison
ax2.plot(history_scratch.history['loss'], label='Scratch - Train')
ax2.plot(history_scratch.history['val_loss'], label='Scratch - Val')
ax2.plot(history_deeper.history['loss'], label='Deeper - Train')
ax2.plot(history_deeper.history['val_loss'], label='Deeper - Val')
ax2.plot(history_pretrained.history['loss'], label='Pretrained - Train')
ax2.plot(history_pretrained.history['val_loss'], label='Pretrained - Val')
ax2.set_title('Model Loss Comparison')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()


# Compute the test accuracy with the three models on the test set, compare the results
# Evaluate all three models on the test set
test_loss_scratch, test_acc_scratch = scratch_model.evaluate(X_test, y_test, verbose=0)
test_loss_deeper, test_acc_deeper = deeper_model.evaluate(X_test, y_test, verbose=0)
test_loss_pretrained, test_acc_pretrained = pre_trained_model.evaluate(X_test, y_test, verbose=0)

# Print the test accuracy results
print("="*50)
print("Test Accuracy Results")
print("="*50)
print(f"Scratch Model Test Accuracy: {test_acc_scratch:.4f} ({test_acc_scratch*100:.2f}%)")
print(f"Deeper Model Test Accuracy: {test_acc_deeper:.4f} ({test_acc_deeper*100:.2f}%)")
print(f"Pretrained Model Test Accuracy: {test_acc_pretrained:.4f} ({test_acc_pretrained*100:.2f}%)")
print("="*50)

# Create a bar chart to visualize the test accuracy comparison
plt.figure(figsize=(10, 6))
models_names = ['Scratch Model', 'Deeper Model', 'Pretrained Model']
test_accuracies = [test_acc_scratch, test_acc_deeper, test_acc_pretrained]
colors = ['skyblue', 'lightgreen', 'salmon']

bars = plt.bar(models_names, test_accuracies, color=colors)
plt.title('Test Accuracy Comparison Across Models')
plt.xlabel('Models')
plt.ylabel('Test Accuracy')
plt.ylim(0, 1)

# Add accuracy values on top of bars
for bar, acc in zip(bars, test_accuracies):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
             f'{acc:.4f}', ha='center', va='bottom', fontweight='bold')

plt.grid(True, axis='y', alpha=0.3)
plt.show()

# Compute predictions for additional metrics
y_pred_scratch = scratch_model.predict(X_test)
y_pred_deeper = deeper_model.predict(X_test)
y_pred_pretrained = pre_trained_model.predict(X_test)

# Convert predictions and true labels to class indices
y_pred_scratch_classes = y_pred_scratch.argmax(axis=1)
y_pred_deeper_classes = y_pred_deeper.argmax(axis=1)
y_pred_pretrained_classes = y_pred_pretrained.argmax(axis=1)
y_true_classes = y_test.argmax(axis=1)

# Calculate accuracy using sklearn
from sklearn.metrics import accuracy_score, classification_report

acc_scratch_sklearn = accuracy_score(y_true_classes, y_pred_scratch_classes)
acc_deeper_sklearn = accuracy_score(y_true_classes, y_pred_deeper_classes)
acc_pretrained_sklearn = accuracy_score(y_true_classes, y_pred_pretrained_classes)

print("\nSklearn Accuracy Confirmation:")
print(f"Scratch Model: {acc_scratch_sklearn:.4f}")
print(f"Deeper Model: {acc_deeper_sklearn:.4f}")
print(f"Pretrained Model: {acc_pretrained_sklearn:.4f}")

# Determine the best performing model
best_acc = max(test_acc_scratch, test_acc_deeper, test_acc_pretrained)
best_model_idx = [test_acc_scratch, test_acc_deeper, test_acc_pretrained].index(best_acc)
best_model_name = models_names[best_model_idx]

print(f"\n🏆 Best performing model: {best_model_name} with accuracy of {best_acc:.4f} ({best_acc*100:.2f}%)")

**What information can you get from these plots?**<br>
**Try to visualize the differences between the deeper model and the pre-trained xception model!
**Are they showing what you expected?**